In [2]:
import ee, json
ee.Authenticate()
ee.Initialize(project="srt-development")

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
years = list(range(2015, 2100))
folder = "SRToutItaly"

In [5]:


import ee, json
from google.colab import drive

ee.Authenticate()
ee.Initialize(project="srt-development")
drive.mount("/content/drive")


# Your grid/table
table = ee.FeatureCollection('projects/srt-development/assets/wildfireItaly/italy_bbox')
gridcoll = ee.FeatureCollection(table)
region = gridcoll.geometry().bounds(1)

# Your dataset
dataset = (ee.ImageCollection("NASA/GDDP-CMIP6")
           .filterBounds(region)
           .filter(ee.Filter.inList("model", ["CMCC-ESM2"]))
           #.filter(ee.Filter.eq('scenario','ssp245'))#############
           .filter(ee.Filter.eq('scenario','ssp585'))
           .select(["pr", "huss", "sfcWind", "tasmax"]))

def annual_stats_image(dataset, year,region_geom):
    y = ee.Number(year)
    ic = dataset.filter(ee.Filter.calendarRange(y, y, "year"))

    reducers = (ee.Reducer.mean()
                .combine(ee.Reducer.stdDev(), sharedInputs=True)
                .combine(ee.Reducer.max(), sharedInputs=True)
                .combine(ee.Reducer.sum(), sharedInputs=True))

    img = ic.reduce(reducers).toFloat().clip(region_geom)

    # deterministic band names: variable-major then stats
    vars_ = ee.List(dataset.first().bandNames())  # respects .select order
    stats_ = ee.List(["mean", "stdDev", "max", "sum"])
    band_names = vars_.map(lambda v: stats_.map(lambda s: ee.String(v).cat("_").cat(s))).flatten()

    return img.rename(band_names).set({"year": y.toInt()})

def fill_nodata_nearestish(img, start_radius=1, max_radius=255):

    img = ee.Image(img)
    filled = img

    r = start_radius
    while r <= max_radius:
        filled = filled.unmask(
            filled.focal_median(radius=r, units='pixels')
        )
        r *= 2

    return filled

def export_annual_multiband(dataset, region_geom, year, out_name_prefix, folder):
    img = annual_stats_image(dataset, year, region_geom)
    native_proj = ee.Image(dataset.first()).select(0).projection()

    img_filled = fill_nodata_nearestish(img, start_radius=1, max_radius=255)

    task = ee.batch.Export.image.toDrive(
        image=img_filled,
        description=f"{out_name_prefix}{int(year)}",
        folder=folder,
        fileNamePrefix=f"{out_name_prefix}{int(year)}",
        region=region_geom,
        crs=native_proj.crs(),
        scale=native_proj.nominalScale(),
        fileFormat="GeoTIFF",
        maxPixels=1e13
    )
    task.start()
    return task

tasks = []
for y in years:
    # Save band names JSON (from the exact image definition used for export)
    band_cols = annual_stats_image(dataset, y, region).bandNames().getInfo()
    band_json_path = f"/content/drive/MyDrive/{folder}/cmip6_bandnames_{y}_ssp585.json"
    with open(band_json_path, "w") as f:
        json.dump(band_cols, f, indent=2)

    print(f"{y} band names saved -> {band_json_path}")
    tasks.append(export_annual_multiband(dataset, region, y, out_name_prefix="cmip6_stats_ssp585", folder=folder))

print("Started", len(tasks), "export tasks for years:", years)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2015 band names saved -> /content/drive/MyDrive/SRToutItaly/cmip6_bandnames_2015_ssp585.json
2016 band names saved -> /content/drive/MyDrive/SRToutItaly/cmip6_bandnames_2016_ssp585.json
2017 band names saved -> /content/drive/MyDrive/SRToutItaly/cmip6_bandnames_2017_ssp585.json
2018 band names saved -> /content/drive/MyDrive/SRToutItaly/cmip6_bandnames_2018_ssp585.json
2019 band names saved -> /content/drive/MyDrive/SRToutItaly/cmip6_bandnames_2019_ssp585.json
2020 band names saved -> /content/drive/MyDrive/SRToutItaly/cmip6_bandnames_2020_ssp585.json
2021 band names saved -> /content/drive/MyDrive/SRToutItaly/cmip6_bandnames_2021_ssp585.json
2022 band names saved -> /content/drive/MyDrive/SRToutItaly/cmip6_bandnames_2022_ssp585.json
2023 band names saved -> /content/drive/MyDrive/SRToutItaly/cmip6_bandnames_2023_ssp585.json
2024 band names saved -> /content/

In [ ]:
import os
import json
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import numpy as np
import geopandas as gpd
import pandas as pd

dst_crs = "EPSG:32633"
target_res = 1000  # meters

points_fp = "/content/drive/MyDrive/SRTout/FireMatrix_centroids32633.gpkg"
base_dir = "/content/drive/MyDrive/SRTout"

def process_year(year: int) -> gpd.GeoDataFrame:
    src_path = os.path.join(base_dir, f"cmip6_stats_ssp585{year}.tif")
    dst_path = os.path.join(base_dir, f"output_1km_{year}ssp585.tif")###

    band_json_path = os.path.join(base_dir, f"cmip6_bandnames_{year}ssp585.json")
    with open(band_json_path, "r") as f:
        band_cols = json.load(f)

    # ---- resample / reproject raster ----
    with rasterio.open(src_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=target_res
        )

        profile = src.profile.copy()
        profile.update({
            "crs": dst_crs,
            "transform": transform,
            "width": width,
            "height": height
        })

        with rasterio.open(dst_path, "w", **profile) as dst:
            for b in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, b),
                    destination=rasterio.band(dst, b),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.bilinear
                )

    # ---- sample points ----
    gdf = gpd.read_file(points_fp)

    with rasterio.open(dst_path) as src:
        if gdf.crs != src.crs:
            gdf = gdf.to_crs(src.crs)

        coords = np.column_stack((gdf.geometry.x, gdf.geometry.y))
        samples = np.vstack(list(src.sample(coords)))  # (n_points, n_bands)

        if samples.shape[1] != len(band_cols):
            raise ValueError(
                f"[{year}] Raster has {samples.shape[1]} bands but band_cols has {len(band_cols)} names"
            )

    # attach sampled band columns
    for i, col in enumerate(band_cols):
        gdf[col] = samples[:, i]

    # keep track of year
    gdf["year"] = year

    return gdf

# ---- run for multiple years and vstack ----
#years = list(range(2000, 2006))  # <-- change to your list, e.g. [2000, 2005, 2010]

gdfs = []
for y in years:
    print("Processing year:", y)
    gdfs.append(process_year(y))

    out_points_fp = os.path.join(base_dir, f"points_sampled_{y}ssp585.gpkg")
    gdfs[-1].to_file(out_points_fp, driver="GPKG")

gdf_all = gpd.GeoDataFrame(
    pd.concat(gdfs, axis=0, ignore_index=True),
    crs=gdfs[0].crs
)



# optional: write combined output
out_all_fp = os.path.join(base_dir, "points_sampled_2549ssp585.gpkg")
gdf_all.to_file(out_all_fp, driver="GPKG")
print("Wrote stacked GPKG:", out_all_fp)


Processing year: 2025
Processing year: 2026
Processing year: 2027
Processing year: 2028
Processing year: 2029
Processing year: 2030
Processing year: 2031
Processing year: 2032
Processing year: 2033
Processing year: 2034
Processing year: 2035
Processing year: 2036
Processing year: 2037
Processing year: 2038
Processing year: 2039
Processing year: 2040
Processing year: 2041
Processing year: 2042
Processing year: 2043
Processing year: 2044
Processing year: 2045
Processing year: 2046
Processing year: 2047
Processing year: 2048
Processing year: 2049
Wrote stacked GPKG: /content/drive/MyDrive/SRTout/points_sampled_2549ssp585.gpkg
